## Active Learning

Active learning is a machine learning technique where the model can interactively query a user (or some other information source) to obtain the desired outputs at new data points. The main idea is to select the most informative data points for labeling, which can help improve the model's performance with fewer labeled examples.

In [18]:
import pandas as pd

df = pd.DataFrame({
    "type": [
        "Berry","Apple","Cherry","Berry","Cherry",
        "Apple","Berry","Cherry","Apple","Berry",
        "Apple","Apple","Pear","Pear","Banana",
        "Banana","Pear","Apple","Berry","Cherry"
    ],
    "size": [
        "Small","Medium","Small","Small","Small",
        "Medium","Small","Small","Small","Medium",
        "Large","Large","Large","Medium","Large",
        "Medium","Small","Medium","Medium","Medium"
    ],
    "color": [
        "Red","Red","Red","Red","Red",
        "Red","Blue","Pink","Red","Red",
        "Green","Yellow","Green","Yellow","Yellow",
        "Yellow","Red","Green","Blue","Pink"
    ],
    "poisonous": [
        "Yes","Yes","Yes","Yes","Yes",
        "Yes","Yes","Yes","Yes","Yes",
        "No","No","No","No","No",
        "No","?","?","?","?"
    ]
})

print(df)

      type    size   color poisonous
0    Berry   Small     Red       Yes
1    Apple  Medium     Red       Yes
2   Cherry   Small     Red       Yes
3    Berry   Small     Red       Yes
4   Cherry   Small     Red       Yes
5    Apple  Medium     Red       Yes
6    Berry   Small    Blue       Yes
7   Cherry   Small    Pink       Yes
8    Apple   Small     Red       Yes
9    Berry  Medium     Red       Yes
10   Apple   Large   Green        No
11   Apple   Large  Yellow        No
12    Pear   Large   Green        No
13    Pear  Medium  Yellow        No
14  Banana   Large  Yellow        No
15  Banana  Medium  Yellow        No
16    Pear   Small     Red         ?
17   Apple  Medium   Green         ?
18   Berry  Medium    Blue         ?
19  Cherry  Medium    Pink         ?


In [19]:
# One-hot encoding for categorical features
encoded_df = pd.get_dummies(df[["type", "size", "color"]]).replace({True: 1, False: 0})

# Add target column
encoded_df["poisonous"] = df["poisonous"]

/tmp/ipykernel_1598/206404578.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encoded_df = pd.get_dummies(df[["type", "size", "color"]]).replace({True: 1, False: 0})


In [20]:
target = "poisonous"
features = encoded_df.columns.drop(target)

# Convert to NumPy arrays
X_encoded = encoded_df[features].values
y = encoded_df[target].values

print("Encoded Features Shape:", X_encoded.shape)
print("Target Shape:", y.shape)

Encoded Features Shape: (20, 13)
Target Shape: (20,)


In [21]:
from sklearn.model_selection import train_test_split

# 25% labeled, 75% unlabeled pool
labeled_set, unlabeled_set = train_test_split(X_encoded, test_size=0.75, random_state=42)

print("Labeled Set Length:", len(labeled_set))
print("Unlabeled Set Length:", len(unlabeled_set))

Labeled Set Length: 5
Unlabeled Set Length: 15


In [22]:
# Split features and labels
X_labeled = labeled_set[:, :-1]
y_labeled = labeled_set[:, -1]

X_unlabeled = unlabeled_set[:, :-1]
y_unlabeled = unlabeled_set[:, -1]  # only for evaluation

In [23]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = LogisticRegression()

In [24]:
import numpy as np

def uncertainty_sampling(model, X_pool, k=5):
    probs = model.predict_proba(X_pool)

    # uncertainty = 1 - max confidence
    uncertainty = 1 - np.max(probs, axis=1)
    k = min(k, len(X_pool))

    # select top-k most uncertain samples
    query_idx = np.argsort(uncertainty)[-k:]

    return query_idx

In [25]:
import numpy as np

iterations = 10

for i in range(iterations):
    print(f"\nIteration {i+1}")

    # Check if there are unlabeled samples remaining
    if len(X_unlabeled) == 0:
        print("No more unlabeled samples to query. Stopping active learning.")
        break

    # Train model on labeled data
    model.fit(X_labeled, y_labeled)

    # Select uncertain samples
    query_idx = uncertainty_sampling(model, X_unlabeled, k=5)

    # Add selected samples to labeled set
    X_labeled = np.vstack((X_labeled, X_unlabeled[query_idx]))
    y_labeled = np.concatenate((y_labeled, y_unlabeled[query_idx]))

    # Remove them from unlabeled pool
    X_unlabeled = np.delete(X_unlabeled, query_idx, axis=0)
    y_unlabeled = np.delete(y_unlabeled, query_idx, axis=0)

    print("Added samples:", len(query_idx))
    print("Labeled set size:", len(X_labeled))
    print("Remaining unlabeled:", len(X_unlabeled))


Iteration 1
Added samples: 5
Labeled set size: 10
Remaining unlabeled: 10

Iteration 2
Added samples: 5
Labeled set size: 15
Remaining unlabeled: 5

Iteration 3
Added samples: 5
Labeled set size: 20
Remaining unlabeled: 0

Iteration 4
No more unlabeled samples to query. Stopping active learning.


In [26]:
if len(X_unlabeled) > 0:
    y_pred = model.predict(X_unlabeled)

    print("\nClassification Report on Remaining Unlabeled Set:")
    print(classification_report(y_unlabeled, y_pred))
else:
    print("\nNo unlabeled samples remaining to evaluate.")


No unlabeled samples remaining to evaluate.
